## Concept 10 — Prompt Optimization & Best Practices

### What is it?
Systematic patterns that consistently produce better LLM outputs.
Prompt engineering is not guesswork — there are repeatable techniques that work.

### The 5 Dimensions of a Strong Prompt
```
1. ROLE        → Who is the LLM?
2. CONTEXT     → What does the LLM need to know?
3. TASK        → What exactly should the LLM do?
4. FORMAT      → How should the output look?
5. CONSTRAINTS → What should the LLM avoid?
```

### What You Will Build
Compare weak vs strong prompts side-by-side across 8 key patterns.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm      = get_llm(temperature=0.3)
llm_det  = get_llm(temperature=0.0)  # deterministic for structured tasks
llm_cre  = get_llm(temperature=0.9)  # creative for brainstorming

def compare(weak_prompt: str, strong_prompt: str, label: str = ''):
    print(f'\n{"="*60}')
    if label:
        print(f'Pattern: {label}')
    print(f'{"="*60}')
    print(f'WEAK:\n  {weak_prompt}')
    print(f'  → {llm.invoke(weak_prompt).content[:200]}')
    print(f'\nSTRONG:\n  {strong_prompt}')
    print(f'  → {llm.invoke(strong_prompt).content[:200]}')

print('Setup complete')

### Pattern 1 — Role Assignment
Assigning a specific expert role produces more focused, accurate answers.

In [ ]:
compare(
    weak_prompt   = 'Explain caching.',
    strong_prompt = 'You are a senior backend engineer. Explain caching to a junior developer in 2 sentences, focusing on why it matters for performance.',
    label         = 'Role Assignment'
)

### Pattern 2 — Chain of Thought
'Think step by step' dramatically improves reasoning for math and logic problems.

In [ ]:
compare(
    weak_prompt   = 'If a train travels 120km in 1.5 hours, what is its speed in m/s?',
    strong_prompt = 'If a train travels 120km in 1.5 hours, what is its speed in m/s? Think step by step.',
    label         = 'Chain of Thought'
)

### Pattern 3 — Output Format Enforcement
Specify the exact output format to get parseable, consistent responses.

In [ ]:
compare(
    weak_prompt   = 'List the top 3 Python web frameworks.',
    strong_prompt = 'List exactly 3 Python web frameworks. Format as:\n1. Name — one sentence description\n2. Name — one sentence description\n3. Name — one sentence description\nNo other text.',
    label         = 'Output Format Enforcement'
)

### Pattern 4 — Negative Constraints (Anti-Hallucination)
Tell the LLM what NOT to do. Reduces hallucination and overconfidence.

In [ ]:
compare(
    weak_prompt   = 'What is the population of Zogonia?',
    strong_prompt = 'What is the population of Zogonia? If you do not know or are unsure, say exactly: I don\'t know. Do NOT make up information.',
    label         = 'Negative Constraints (Anti-Hallucination)'
)

### Pattern 5 — Grounding (RAG-style Constraint)
Force the LLM to answer ONLY from provided context — the core RAG safety pattern.

In [ ]:
context = 'Python lists are ordered, mutable collections that allow duplicates. Tuples are ordered and immutable.'

grounded_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Answer ONLY from the provided context. '
     'If the answer is not in the context, say: I don\'t know. '
     'Context: {context}'),
    ('human', '{question}'),
])
grounded_chain = grounded_prompt | llm_det | StrOutputParser()

questions = [
    'Can a Python list have duplicates?',
    'What is a Python dictionary?',  # not in context — should say I don't know
]

print('Grounded (context-only) answers:')
for q in questions:
    ans = grounded_chain.invoke({'context': context, 'question': q})
    print(f'  Q: {q}')
    print(f'  A: {ans}\n')

### Pattern 6 — Temperature vs Task Type
Temperature controls randomness. Different tasks need different settings.

In [ ]:
task = 'Give me 3 creative names for a Python learning app.'

print(f'Task: {task}\n')

for temp, label in [(0.0, 'temp=0.0 (deterministic)'), (0.7, 'temp=0.7 (balanced)'), (1.2, 'temp=1.2 (creative)')]:
    model = get_llm(temperature=temp)
    result = model.invoke(task).content
    print(f'[{label}]\n{result}\n')

### Pattern 7 — Token Efficiency
Verbose system prompts cost tokens without improving quality. Keep them tight.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

verbose_system = (
    'You are a very helpful and knowledgeable assistant that is always ready to '
    'assist the user with any question they might have, no matter how simple or '
    'complex it might be, and you always try your very best to give thorough answers.'
)
tight_system = 'You are a helpful assistant.'

question = 'What is a REST API?'

verbose_resp = llm.invoke([SystemMessage(content=verbose_system), HumanMessage(content=question)])
tight_resp   = llm.invoke([SystemMessage(content=tight_system),   HumanMessage(content=question)])

print(f'Verbose system ({len(verbose_system)} chars):\n  {verbose_resp.content[:200]}\n')
print(f'Tight system ({len(tight_system)} chars):\n  {tight_resp.content[:200]}')
print(f'\nSaved ~{len(verbose_system)-len(tight_system)} chars of tokens — same quality!')

### Pattern 8 — Self-Consistency
For critical decisions, generate multiple answers and pick the consensus.

In [ ]:
consistency_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a code reviewer.'),
    ('human',
     'Review this function for bugs. Give 3 independent assessments, '
     'then state the consensus finding.\n\n'
     'Code:\n```python\n{code}\n```'),
])
consistency_chain = consistency_prompt | llm_cre | StrOutputParser()

buggy_code = """
def average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)
"""

result = consistency_chain.invoke({'code': buggy_code})
print(result)

### Summary — Quick Reference

In [ ]:
summary = {
    'Role Assignment':    'You are a [expert]. → More focused, domain-accurate answers',
    'Chain of Thought':   'Think step by step. → Better reasoning for math/logic',
    'Format Enforcement': 'Output ONLY as X. → Consistent, parseable responses',
    'Negative Constraints': 'Do NOT make up info. → Reduces hallucination',
    'Grounding':          'Answer ONLY from context. → RAG safety pattern',
    'Temperature':        '0=deterministic, 0.7=balanced, 1.2+=creative',
    'Token Efficiency':   'Tight system prompts = same quality, lower cost',
    'Self-Consistency':   'Generate N answers → pick consensus = more reliable',
}

print('Prompt Optimization Cheat Sheet:')
print('-' * 60)
for pattern, description in summary.items():
    print(f'{pattern:<22} {description}')